#Ingest drivers json file
-  Read the file using spark dataframe reader api
- Definf and enforce Schema(prserve nested structure)
- Add metadata columns ->Source File -> Ingestion timestamp
- Write to Bronze delta table

In [0]:
%run ../00-common/01.environment.config

In [0]:
%run ../00-common/02.bronze_helper

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema_name}.drivers"

#Step 1 - Read the file using spark dataframe reader api

In [0]:
#Define schema
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName',StringType()),
    StructField('familyName',StringType())
])

drivers_schema = StructType([
    StructField('driverId',StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth',DateType()),
    StructField('nationality',StringType()),
    StructField('url', StringType())
])


   

In [0]:
#read data from json file
drivers_df = (spark.read
                   .format("json")
                   .schema(drivers_schema)
                   .option('mode', 'FAILFAST')
                   .load(source_file)
                   )


In [0]:
display(drivers_df)

driverId,name,dateOfBirth,nationality,url
abate,"List(carlo, abate)",1932-07-10,italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate
abecassis,"List(george, abecassis)",1913-03-21,british,http://en.wikipedia.org/wiki/George_Abecassis
acheson,"List(kenny, acheson)",1957-11-27,british,http://en.wikipedia.org/wiki/Kenny_Acheson
adams,"List(philippe, adams)",1969-11-19,belgian,http://en.wikipedia.org/wiki/Philippe_Adams
ader,"List(walt, ader)",1913-12-15,american,http://en.wikipedia.org/wiki/Walt_Ader
adolff,"List(kurt, adolff)",1921-11-05,german,http://en.wikipedia.org/wiki/Kurt_Adolff
agabashian,"List(fred, agabashian)",1913-08-21,american,http://en.wikipedia.org/wiki/Fred_Agabashian
ahrens,"List(kurt, ahrens)",1940-04-19,german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr."
aitken,"List(jack, aitken)",1995-09-23,british,http://en.wikipedia.org/wiki/Jack_Aitken
albers,"List(christijan, albers)",1979-04-16,dutch,http://en.wikipedia.org/wiki/Christijan_Albers


#Step 2 - Add Metadata

In [0]:
drivers_final_df = add_ingestion_metadata(drivers_df)

#Step 3 - Write to bronze delta table

In [0]:
(
    drivers_final_df
    .write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))

driverId,name,dateOfBirth,nationality,url,ingestion_timestamp,source_file
abate,"List(carlo, abate)",1932-07-10,italian,http://en.wikipedia.org/wiki/Carlo_Mario_Abate,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
abecassis,"List(george, abecassis)",1913-03-21,british,http://en.wikipedia.org/wiki/George_Abecassis,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
acheson,"List(kenny, acheson)",1957-11-27,british,http://en.wikipedia.org/wiki/Kenny_Acheson,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
adams,"List(philippe, adams)",1969-11-19,belgian,http://en.wikipedia.org/wiki/Philippe_Adams,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
ader,"List(walt, ader)",1913-12-15,american,http://en.wikipedia.org/wiki/Walt_Ader,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
adolff,"List(kurt, adolff)",1921-11-05,german,http://en.wikipedia.org/wiki/Kurt_Adolff,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
agabashian,"List(fred, agabashian)",1913-08-21,american,http://en.wikipedia.org/wiki/Fred_Agabashian,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
ahrens,"List(kurt, ahrens)",1940-04-19,german,"http://en.wikipedia.org/wiki/Kurt_Ahrens,_Jr.",2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
aitken,"List(jack, aitken)",1995-09-23,british,http://en.wikipedia.org/wiki/Jack_Aitken,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
albers,"List(christijan, albers)",1979-04-16,dutch,http://en.wikipedia.org/wiki/Christijan_Albers,2026-06-16T14:03:14.014812Z,dbfs:/Volumes/formula1/landing/files/drivers.json
